# 03 - Data Augmentation Offline

Este notebook parte das imagens tratadas geradas em `02_preprocessing.ipynb`, faz o downsampling da classe negativa, cria o split `train/val/test` e exporta os datasets finais para comparacao justa entre baseline e augmentation.

**Objetivos**

- carregar o dataset tratado em `data/processed/treated/`
- fazer downsampling da classe negativa e split estratificado `train/val/test`
- exportar os splits sem augmentation:
  - `data/processed/without_augmentation/`
  - `data/processed/without_augmentation_64x64/`
- gerar copias aumentadas apenas no split de treino
- exportar tambem:
  - `data/processed/with_augmentation/`
  - `data/processed/with_augmentation_64x64/`
        

In [8]:
import json
import random
import shutil
import warnings
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")


def resolve_repo_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar a raiz com data/processed.")


ROOT_DIR = resolve_repo_root()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TREATED_DIR = PROCESSED_DIR / "treated"
TREATED_MANIFEST = PROCESSED_DIR / "treated_manifest.csv"
OUTPUT_DIR = ROOT_DIR / "notebooks" / "outputs"
PREP_DIR = OUTPUT_DIR / "preprocessing"
FIG_DIR = OUTPUT_DIR / "figures"

for directory in [PREP_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
NON_MELANOMA_TO_MELANOMA_RATIO = 3.0
AUGMENTED_EXPORT_COPIES = 2
INCLUDE_ORIGINAL_IN_AUGMENTED_TRAIN = True
EXPORT_IMAGE_FORMAT = "png"
BINARY_CLASS_DIRS = {0: "0_non_melanoma", 1: "1_melanoma"}

EXPORT_BASELINE_NAME = "without_augmentation"
EXPORT_AUGMENTED_NAME = "with_augmentation"
EXPORT_BASELINE_NAME_64 = "without_augmentation_64x64"
EXPORT_AUGMENTED_NAME_64 = "with_augmentation_64x64"
EXPORT_VARIANTS = [
    {
        "baseline_name": EXPORT_BASELINE_NAME,
        "augmented_name": EXPORT_AUGMENTED_NAME,
        "target_size": 224,
    },
    {
        "baseline_name": EXPORT_BASELINE_NAME_64,
        "augmented_name": EXPORT_AUGMENTED_NAME_64,
        "target_size": 64,
    },
]

config = {
    "seed": SEED,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "non_melanoma_to_melanoma_ratio": NON_MELANOMA_TO_MELANOMA_RATIO,
    "augmented_export_copies": AUGMENTED_EXPORT_COPIES,
    "include_original_in_augmented_train": INCLUDE_ORIGINAL_IN_AUGMENTED_TRAIN,
    "export_image_format": EXPORT_IMAGE_FORMAT,
    "export_baseline_name": EXPORT_BASELINE_NAME,
    "export_augmented_name": EXPORT_AUGMENTED_NAME,
    "export_baseline_name_64": EXPORT_BASELINE_NAME_64,
    "export_augmented_name_64": EXPORT_AUGMENTED_NAME_64,
    "export_variants": EXPORT_VARIANTS,
}

print(json.dumps(config, indent=2))
        

{
  "seed": 42,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  "non_melanoma_to_melanoma_ratio": 3.0,
  "augmented_export_copies": 2,
  "include_original_in_augmented_train": true,
  "export_image_format": "png",
  "export_baseline_name": "without_augmentation",
  "export_augmented_name": "with_augmentation",
  "export_baseline_name_64": "without_augmentation_64x64",
  "export_augmented_name_64": "with_augmentation_64x64",
  "export_variants": [
    {
      "baseline_name": "without_augmentation",
      "augmented_name": "with_augmentation",
      "target_size": 224
    },
    {
      "baseline_name": "without_augmentation_64x64",
      "augmented_name": "with_augmentation_64x64",
      "target_size": 64
    }
  ]
}


## 1. Carregamento do dataset tratado

Carrega o manifesto gerado pelo `02_preprocessing.ipynb` a partir de `data/processed/treated/`.
        

In [9]:
if not TREATED_MANIFEST.exists():
    raise FileNotFoundError(
        f"Manifesto nao encontrado em {TREATED_MANIFEST}. "
        "Execute o notebook 02_preprocessing.ipynb primeiro."
    )
if not TREATED_DIR.exists():
    raise FileNotFoundError(
        f"Pasta de imagens tratadas nao encontrada em {TREATED_DIR}. "
        "Execute o notebook 02_preprocessing.ipynb primeiro."
    )

treated_df = pd.read_csv(TREATED_MANIFEST).copy()

required_columns = {"image_id", "binary_label", "label", "export_path"}
missing_columns = required_columns - set(treated_df.columns)
if missing_columns:
    raise ValueError(f"treated_manifest.csv sem colunas obrigatorias: {sorted(missing_columns)}")

treated_summary = pd.DataFrame(
    [
        {"metrica": "total_imagens_tratadas", "valor": len(treated_df)},
        {"metrica": "melanomas", "valor": int((treated_df["binary_label"] == 1).sum())},
        {"metrica": "nao_melanomas", "valor": int((treated_df["binary_label"] == 0).sum())},
        {
            "metrica": "ratio_bruto",
            "valor": round(
                (treated_df["binary_label"] == 0).sum() / max(int((treated_df["binary_label"] == 1).sum()), 1),
                2,
            ),
        },
    ]
)

display(treated_summary)
print(f"Dataset tratado carregado: {len(treated_df):,} imagens de {TREATED_DIR}")
        

,metrica,valor
0,total_imagens_tratadas,10015.0
1,melanomas,1113.0
2,nao_melanomas,8902.0
3,ratio_bruto,8.0


Dataset tratado carregado: 10,015 imagens de /Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/data/processed/treated


## 2. Downsampling e Split train / val / test

Reduz a classe negativa para o ratio configuravel e divide em `train/val/test` de forma estratificada. Esse mesmo split eh exportado tanto para as branches baseline quanto para as branches com augmentation.
        

In [10]:
def build_effective_dataset(df: pd.DataFrame, non_melanoma_ratio: float, seed: int) -> pd.DataFrame:
    melanoma_df = df[df["binary_label"] == 1].copy()
    non_melanoma_df = df[df["binary_label"] == 0].copy()

    target_non_melanoma = min(
        len(non_melanoma_df),
        int(round(len(melanoma_df) * non_melanoma_ratio)),
    )

    if target_non_melanoma <= 0:
        raise ValueError("O ratio solicitado produziu um subconjunto negativo vazio.")

    if target_non_melanoma == len(non_melanoma_df):
        selected_non_melanoma = non_melanoma_df.copy()
    else:
        selected_non_melanoma, _ = train_test_split(
            non_melanoma_df,
            train_size=target_non_melanoma,
            stratify=non_melanoma_df["label"],
            random_state=seed,
        )

    melanoma_df["selection_source"] = "kept_all_melanoma"
    selected_non_melanoma["selection_source"] = "downsampled_non_melanoma"

    return (
        pd.concat([melanoma_df, selected_non_melanoma], axis=0)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )


df = build_effective_dataset(treated_df, NON_MELANOMA_TO_MELANOMA_RATIO, SEED)

train_df, temp_df = train_test_split(
    df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=df["binary_label"],
    random_state=SEED,
)

relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
val_df, test_df = train_test_split(
    temp_df,
    test_size=relative_test_ratio,
    stratify=temp_df["binary_label"],
    random_state=SEED,
)

split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "melanoma": int((split_frame["binary_label"] == 1).sum()),
            "nao_melanoma": int((split_frame["binary_label"] == 0).sum()),
            "total": len(split_frame),
        }
        for split_name, split_frame in [("train", train_df), ("val", val_df), ("test", test_df)]
    ]
)

display(split_summary)
print(f"Split concluido: train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}")
print(f"Ratio efetivo: {(df['binary_label'] == 0).sum() / max(int((df['binary_label'] == 1).sum()), 1):.2f}:1")
        

,split,melanoma,nao_melanoma,total
0,train,779,2337,3116
1,val,167,501,668
2,test,167,501,668


Split concluido: train=3,116, val=668, test=668
Ratio efetivo: 3.00:1


## 3. Pipeline de augmentation
        

In [11]:
try:
    import albumentations as A
except ImportError as exc:
    raise ImportError("Albumentations e necessario para a etapa de augmentation.") from exc


offline_aug_export_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.15),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.03,
            scale_limit=0.08,
            rotate_limit=20,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.7,
        ),
        A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.5),
    ]
)

print("Pipeline de augmentation definido.")
        

Pipeline de augmentation definido.


## 4. Pre-visualizacao rapida
        

In [12]:
train_preview = train_df.sample(1, random_state=SEED).iloc[0]
preview_image_raw = np.array(Image.open(train_preview["export_path"]).convert("RGB"))
preview_image = cv2.resize(preview_image_raw, (224, 224), interpolation=cv2.INTER_AREA)

preview_augmented = [offline_aug_export_transform(image=preview_image)["image"] for _ in range(3)]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(preview_image)
axes[0].set_title(f"original ({train_preview['label']})")
axes[0].axis("off")

for idx, aug_image in enumerate(preview_augmented, start=1):
    axes[idx].imshow(aug_image)
    axes[idx].set_title(f"aug_{idx}")
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "augmentation_preview.png", dpi=150, bbox_inches="tight")
plt.show()
        

## 5. Exportacao dos datasets baseline e augmented

Para cada tamanho alvo:

- `without_augmentation`: mesmo split `train/val/test`, apenas resize
- `with_augmentation`: mesmo split, com augmentation aplicada apenas no treino
        

In [13]:
def save_rgb_image(image_array: np.ndarray, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(image_array.astype(np.uint8)).save(output_path)


def prepare_export_root(export_root: Path) -> None:
    if export_root.exists():
        shutil.rmtree(export_root)
    export_root.mkdir(parents=True, exist_ok=True)


def export_split(
    frame: pd.DataFrame,
    split_name: str,
    export_root: Path,
    target_size: int,
    augmenter=None,
    augmented_copies: int = 0,
    include_original: bool = True,
) -> pd.DataFrame:
    manifest_rows = []
    total = len(frame)

    for idx, row in enumerate(frame.itertuples(), start=1):
        image_raw = np.array(Image.open(row.export_path).convert("RGB"))
        image = cv2.resize(image_raw, (target_size, target_size), interpolation=cv2.INTER_AREA)
        class_dir = export_root / split_name / BINARY_CLASS_DIRS[row.binary_label]

        if include_original or augmenter is None:
            output_path = class_dir / f"{row.image_id}.{EXPORT_IMAGE_FORMAT}"
            save_rgb_image(image, output_path)
            manifest_rows.append(
                {
                    "image_id": row.image_id,
                    "split": split_name,
                    "binary_label": row.binary_label,
                    "label": row.label,
                    "selection_source": row.selection_source,
                    "is_augmented": False,
                    "augmentation_copy": 0,
                    "export_path": str(output_path),
                }
            )

        if augmenter is not None:
            for copy_idx in range(1, augmented_copies + 1):
                aug_image = augmenter(image=image)["image"]
                aug_path = class_dir / f"{row.image_id}_aug{copy_idx}.{EXPORT_IMAGE_FORMAT}"
                save_rgb_image(aug_image, aug_path)
                manifest_rows.append(
                    {
                        "image_id": row.image_id,
                        "split": split_name,
                        "binary_label": row.binary_label,
                        "label": row.label,
                        "selection_source": row.selection_source,
                        "is_augmented": True,
                        "augmentation_copy": copy_idx,
                        "export_path": str(aug_path),
                    }
                )

        if idx % 500 == 0 or idx == total:
            print(f"    {idx}/{total} imagens processadas ({split_name})...")

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(export_root / f"{split_name}_manifest.csv", index=False)
    return manifest_df


baseline_results = {}
augmented_results = {}

for variant in EXPORT_VARIANTS:
    baseline_name = variant["baseline_name"]
    augmented_name = variant["augmented_name"]
    target_size = variant["target_size"]
    baseline_root = PROCESSED_DIR / baseline_name
    augmented_root = PROCESSED_DIR / augmented_name

    print(f"\nExportando {baseline_name} ({target_size}x{target_size})...")
    prepare_export_root(baseline_root)
    baseline_manifests = {
        "train": export_split(train_df, "train", baseline_root, target_size),
        "val": export_split(val_df, "val", baseline_root, target_size),
        "test": export_split(test_df, "test", baseline_root, target_size),
    }
    pd.concat(baseline_manifests.values(), axis=0).to_csv(baseline_root / "full_manifest.csv", index=False)
    baseline_results[baseline_name] = {
        "target_size": target_size,
        "root": baseline_root,
        "manifests": baseline_manifests,
    }

    print(f"\nExportando {augmented_name} ({target_size}x{target_size})...")
    prepare_export_root(augmented_root)
    augmented_manifests = {
        "train": export_split(
            train_df,
            "train",
            augmented_root,
            target_size,
            augmenter=offline_aug_export_transform,
            augmented_copies=AUGMENTED_EXPORT_COPIES,
            include_original=INCLUDE_ORIGINAL_IN_AUGMENTED_TRAIN,
        ),
        "val": export_split(val_df, "val", augmented_root, target_size),
        "test": export_split(test_df, "test", augmented_root, target_size),
    }
    pd.concat(augmented_manifests.values(), axis=0).to_csv(augmented_root / "full_manifest.csv", index=False)
    augmented_results[augmented_name] = {
        "target_size": target_size,
        "root": augmented_root,
        "manifests": augmented_manifests,
    }

config["output_paths"] = {
    **{name: str(data["root"]) for name, data in baseline_results.items()},
    **{name: str(data["root"]) for name, data in augmented_results.items()},
}
(PREP_DIR / "augmentation_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

export_summary = pd.DataFrame(
    [
        {
            "experimento": name,
            "target_size": data["target_size"],
            "train_images": len(data["manifests"]["train"]),
            "val_images": len(data["manifests"]["val"]),
            "test_images": len(data["manifests"]["test"]),
        }
        for name, data in baseline_results.items()
    ]
    + [
        {
            "experimento": name,
            "target_size": data["target_size"],
            "train_images": len(data["manifests"]["train"]),
            "val_images": len(data["manifests"]["val"]),
            "test_images": len(data["manifests"]["test"]),
        }
        for name, data in augmented_results.items()
    ]
)

display(export_summary)
print("\nExportacao concluida.")
        


Exportando without_augmentation (224x224)...
    500/3116 imagens processadas (train)...
    1000/3116 imagens processadas (train)...
    1500/3116 imagens processadas (train)...
    2000/3116 imagens processadas (train)...
    2500/3116 imagens processadas (train)...
    3000/3116 imagens processadas (train)...
    3116/3116 imagens processadas (train)...
    500/668 imagens processadas (val)...
    668/668 imagens processadas (val)...
    500/668 imagens processadas (test)...
    668/668 imagens processadas (test)...

Exportando with_augmentation (224x224)...
    500/3116 imagens processadas (train)...
    1000/3116 imagens processadas (train)...
    1500/3116 imagens processadas (train)...
    2000/3116 imagens processadas (train)...
    2500/3116 imagens processadas (train)...
    3000/3116 imagens processadas (train)...
    3116/3116 imagens processadas (train)...
    500/668 imagens processadas (val)...
    668/668 imagens processadas (val)...
    500/668 imagens processadas (te

,experimento,target_size,train_images,val_images,test_images
0,without_augmentation,224,3116,668,668
1,without_augmentation_64x64,64,3116,668,668
2,with_augmentation,224,9348,668,668
3,with_augmentation_64x64,64,9348,668,668



Exportacao concluida.


## 6. Conclusoes
        

In [14]:
baseline_train_manifest = baseline_results[EXPORT_BASELINE_NAME]["manifests"]["train"]
augmented_train_manifest = augmented_results[EXPORT_AUGMENTED_NAME]["manifests"]["train"]

baseline_binary = baseline_train_manifest["binary_label"].value_counts().sort_index()
augmented_binary = augmented_train_manifest["binary_label"].value_counts().sort_index()

baseline_nonmel = int(baseline_binary.get(0, 0))
baseline_mel = int(baseline_binary.get(1, 0))
augmented_nonmel = int(augmented_binary.get(0, 0))
augmented_mel = int(augmented_binary.get(1, 0))
augmented_total = len(augmented_train_manifest)
augmented_original = int((~augmented_train_manifest["is_augmented"]).sum())
augmented_copies = int(augmented_train_manifest["is_augmented"].sum())

summary_table = pd.DataFrame(
    [
        {"metrica": "train_export_baseline", "valor": len(baseline_train_manifest)},
        {"metrica": "train_export_augmented", "valor": augmented_total},
        {"metrica": "baseline_train_melanoma", "valor": baseline_mel},
        {"metrica": "baseline_train_nao_melanoma", "valor": baseline_nonmel},
        {"metrica": "augmented_train_melanoma", "valor": augmented_mel},
        {"metrica": "augmented_train_nao_melanoma", "valor": augmented_nonmel},
    ]
)
display(summary_table)

print("Conclusoes:")
print(
    f"- O split sem augmentation foi exportado em `{EXPORT_BASELINE_NAME}` e `{EXPORT_BASELINE_NAME_64}` "
    f"com {len(baseline_train_manifest)} imagens no treino."
)
print(
    f"- A branch `{EXPORT_AUGMENTED_NAME}` reutiliza o mesmo split e expande apenas o treino para "
    f"{augmented_total} imagens ({augmented_original} originais + {augmented_copies} augmentadas)."
)
print(
    f"- A proporcao entre classes foi preservada: {baseline_nonmel / max(baseline_mel, 1):.2f}:1 no baseline "
    f"e {augmented_nonmel / max(augmented_mel, 1):.2f}:1 com augmentation."
)
print(
    f"- O mesmo fluxo tambem gera `{EXPORT_AUGMENTED_NAME_64}` a partir do split exportado em "
    f"`{EXPORT_BASELINE_NAME_64}`."
)
        

,metrica,valor
0,train_export_baseline,3116
1,train_export_augmented,9348
2,baseline_train_melanoma,779
3,baseline_train_nao_melanoma,2337
4,augmented_train_melanoma,2337
5,augmented_train_nao_melanoma,7011


Conclusoes:
- O split sem augmentation foi exportado em `without_augmentation` e `without_augmentation_64x64` com 3116 imagens no treino.
- A branch `with_augmentation` reutiliza o mesmo split e expande apenas o treino para 9348 imagens (3116 originais + 6232 augmentadas).
- A proporcao entre classes foi preservada: 3.00:1 no baseline e 3.00:1 com augmentation.
- O mesmo fluxo tambem gera `with_augmentation_64x64` a partir do split exportado em `without_augmentation_64x64`.
